# 🛒 eBay Scraping Agent — Jupyter Notebook

**Automatisierte eBay-Produktsuche, Preisanalyse und Preisalarme**

Dieses Notebook demonstriert die Kernfunktionen des eBay Scraping Agents:
- 🔍 eBay-Suche mit Browser-Use
- 📊 Preisanalyse (Statistiken, Verteilung, Deal-Erkennung)
- 🔔 Preisalarme mit Schwellwerten
- 📦 Export als JSON/CSV

---

## 1. Setup & Importe

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import List, Optional

# Repository-Pfad hinzufügen
sys.path.insert(0, '/opt/data/ebay-scraping-agent')

# Versuche, den echten Scraper zu importieren
try:
    from ebay_scraper import EbayProduct
    print('✅ EbayProduct-Dataclass importiert')
except ImportError:
    # Fallback: Eigene Dataclass definieren
    @dataclass
    class EbayProduct:
        title: str
        price: float
        condition: str
        shipping: float
        url: str
        seller: str = ''
        location: str = ''
    print('⚠️ Fallback-Dataclass verwendet')

print(f'📅 Ausführungszeit: {datetime.now().strftime("%d.%m.%Y %H:%M")}')
print(f'🐍 Python: {sys.version}')

## 2. Demo-Produktdaten generieren

Simuliert realistische eBay-Suchergebnisse für Tests und Analyse.

In [ ]:
def generate_demo_products(query: str, count: int = 12) -> List[EbayProduct]:
    """Erzeugt realistische Demo-Produktdaten basierend auf der Suchanfrage."""
    conditions = ['Neu', 'Gebraucht', 'Neu', 'Gebraucht', 'Neu', 'Generalüberholt']
    sellers = ['top-seller-24', 'electronics-pro', 'private-verkauf', 'outlet-store',
               'tech-deals', 'gadget-world', 'spar-fuchs', 'premium-shop']
    locations = ['Berlin', 'München', 'Hamburg', 'Köln', 'Frankfurt', 'Stuttgart']
    
    products = []
    base_price = np.random.uniform(50, 500)
    
    for i in range(count):
        variant = f'{query} - Variante {i+1}'
        price = base_price * np.random.uniform(0.6, 1.5)
        shipping = np.random.choice([0.0, 0.0, 0.0, 4.99, 5.99, 6.99])
        products.append(EbayProduct(
            title=variant,
            price=round(price, 2),
            condition=np.random.choice(conditions),
            shipping=shipping,
            url=f'https://ebay.de/itm/example{i+1}',
            seller=np.random.choice(sellers),
            location=np.random.choice(locations),
        ))
    
    return sorted(products, key=lambda p: p.price)

# Demo-Suche ausführen
SEARCH_QUERY = 'iPhone 15'
products = generate_demo_products(SEARCH_QUERY, count=15)
print(f'🔍 Suche nach {SEARCH_QUERY}: {len(products)} Produkte gefunden\n')

# Erste 5 Produkte anzeigen
df_preview = pd.DataFrame([{
    'Titel': p.title,
    'Preis (€)': p.price,
    'Zustand': p.condition,
    'Versand (€)': p.shipping,
    'Verkäufer': p.seller,
    'Standort': p.location,
} for p in products[:5]])
df_preview

## 3. Preisanalyse

Berechnung von Preisstatistiken: Min, Max, Durchschnitt, Median, Standardabweichung.

In [ ]:
def analyze_prices(products: List[EbayProduct]) -> dict:
    """Preisstatistiken berechnen."""
    if not products:
        return {}
    prices = [p.price for p in products if p.price > 0]
    if not prices:
        return {}
    arr = np.array(prices)
    return {
        'Anzahl': len(arr),
        'Min (€)': round(float(arr.min()), 2),
        'Max (€)': round(float(arr.max()), 2),
        'Durchschnitt (€)': round(float(arr.mean()), 2),
        'Median (€)': round(float(np.median(arr)), 2),
        'Std-Abw. (€)': round(float(arr.std()), 2),
    }

stats = analyze_prices(products)
print('📊 Preisstatistik:')
for key, value in stats.items():
    print(f'  {key}: {value}')

# Als DataFrame
pd.DataFrame([stats]).T.rename(columns={0: 'Wert'})

## 4. Preisverteilung visualisieren

In [ ]:
import matplotlib.pyplot as plt

prices = [p.price for p in products]
titles = [p.title[:25] + '...' for p in products]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Balkendiagramm
colors = ['#4CAF50' if p < np.mean(prices) else '#FF9800' for p in prices]
ax1.barh(range(len(prices)), prices, color=colors)
ax1.set_yticks(range(len(prices)))
ax1.set_yticklabels(titles, fontsize=8)
ax1.set_xlabel('Preis (€)')
ax1.set_title(f'Preisverteilung: {SEARCH_QUERY}')
ax1.axvline(np.mean(prices), color='red', linestyle='--', label=f'Ø {np.mean(prices):.2f} €')
ax1.legend()

# Histogramm
ax2.hist(prices, bins=10, color='#2196F3', edgecolor='white', alpha=0.8)
ax2.axvline(np.mean(prices), color='red', linestyle='--', label=f'Ø {np.mean(prices):.2f} €')
ax2.axvline(np.median(prices), color='green', linestyle='--', label=f'Median {np.median(prices):.2f} €')
ax2.set_xlabel('Preis (€)')
ax2.set_ylabel('Anzahl')
ax2.set_title('Preis-Histogramm')
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Preisspanne nach Zustand

In [ ]:
df_cond = pd.DataFrame([{
    'Zustand': p.condition,
    'Preis (€)': p.price,
} for p in products])

print('📦 Durchschnittspreis nach Zustand:\n')
summary = df_cond.groupby('Zustand').agg(
    Anzahl=('Preis (€)', 'count'),
    Durchschnitt=('Preis (€)', 'mean'),
    Minimum=('Preis (€)', 'min'),
    Maximum=('Preis (€)', 'max'),
).round(2)
summary

## 6. Deal-Finder (Preisalarm)

Findet Produkte unter einem bestimmten Zielpreis.

In [ ]:
TARGET_PRICE = 200.0  # Zielpreis in €

bargains = [p for p in products if p.price <= TARGET_PRICE]

print(f'🎯 Zielpreis: {TARGET_PRICE:.2f} €')
print(f'🎉 {len(bargains)} Schnäppchen gefunden!\n')

if bargains:
    df_bargains = pd.DataFrame([{
        'Titel': p.title,
        'Preis (€)': f'{p.price:.2f}',
        'Ersparnis (€)': f'{TARGET_PRICE - p.price:.2f}',
        'Zustand': p.condition,
        'Versand (€)': f'{p.shipping:.2f}',
        'URL': p.url,
    } for p in bargains])
    df_bargains
else:
    print(f'Keine Produkte unter {TARGET_PRICE:.2f} € gefunden.')

## 7. Erweiterte Filter

Filter nach Zustand, Maximalpreis und Versandkosten.

In [ ]:
# Filter-Parameter
CONDITION_FILTER = ['Neu', 'Gebraucht']
MAX_PRICE = 500.0
FREE_SHIPPING_ONLY = True

filtered = [
    p for p in products
    if p.condition in CONDITION_FILTER
    and p.price <= MAX_PRICE
    and (not FREE_SHIPPING_ONLY or p.shipping == 0.0)
]

print(f'🔍 Filter: Zustand={CONDITION_FILTER}, Max-Preis={MAX_PRICE}€, Gratis-Versand={FREE_SHIPPING_ONLY}')
print(f'📋 {len(filtered)} von {len(products)} Produkten entsprechen den Kriterien\n')

if filtered:
    df_filtered = pd.DataFrame([{
        'Titel': p.title,
        'Preis (€)': f'{p.price:.2f}',
        'Zustand': p.condition,
        'Versand (€)': f'{p.shipping:.2f}',
        'Verkäufer': p.seller,
    } for p in filtered])
    df_filtered
else:
    print('Keine Produkte entsprechen den Filterkriterien.')

## 8. Export als JSON & CSV

In [ ]:
# JSON-Export
data = [asdict(p) for p in products]
json_str = json.dumps(data, indent=2, ensure_ascii=False)

export_dir = '/opt/data/ebay-scraping-agent/exports'
os.makedirs(export_dir, exist_ok=True)

json_path = os.path.join(export_dir, f'ebay_{SEARCH_QUERY.replace(" ", "_")}_{datetime.now().strftime("%Y%m%d")}.json')
with open(json_path, 'w', encoding='utf-8') as f:
    f.write(json_str)
print(f'✅ JSON exportiert: {json_path} ({len(json_str)} Zeichen)')

# CSV-Export
df_export = pd.DataFrame(data)
csv_path = os.path.join(export_dir, f'ebay_{SEARCH_QUERY.replace(" ", "_")}_{datetime.now().strftime("%Y%m%d")}.csv')
df_export.to_csv(csv_path, index=False)
print(f'✅ CSV exportiert: {csv_path} ({len(df_export)} Zeilen)')

# Vorschau
print(f'\n📋 JSON-Vorschau (erste 200 Zeichen):')
print(json_str[:200] + '...')

## 9. Zusammenfassung

| Funktion | Beschreibung |
|---|---|
| 🔍 **Suche** | eBay-Produktsuche mit Browser-Use |
| 📊 **Analyse** | Min/Max/Median, Preisverteilung, Zustandsanalyse |
| 🔔 **Alarme** | Schwellwert-basierte Deal-Erkennung |
| 🔧 **Filter** | Zustand, Max-Preis, Gratis-Versand |
| 📦 **Export** | JSON & CSV |

---

**Nächste Schritte:**
- Echten Browser-Use-Scraper mit `ebay_scraper.py` verwenden
- Streamlit-App starten: `streamlit run app/app.py`
- Preisalarme mit E-Mail-Benachrichtigung konfigurieren